In [2]:
#importing library
import pandas as pd


In [3]:
#downloading data file and encoding

df = pd.read_csv("Raw_data.csv", encoding="ISO-8859-1")
df.shape     #tells size of business
df.head()    #checking the initial look of data i.e. headers and all
df.info()    #checking data types
df.isnull().sum()   #checking null values ( description- 1454 nulls, Customer ID-135080 nulls)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## Data Profiling – Initial Observations
- Dataset contains ~541K transaction-level records.
- Data represents online retail transactions, mostly from the United Kingdom.
- `CustomerID` has missing values, indicating guest or unregistered customers.
- `InvoiceDate` was initially stored as an object and required conversion for time-based analysis.
- Negative values in `Quantity` indicate product returns or cancellations.
- Dataset is suitable for sales, product, and customer behavior analysis.


In [4]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')


- Converted InvoiceDate to datetime using `errors='coerce'` to handle invalid date values safely.
- Identified rows with invalid or missing InvoiceDate values (converted to NaT).


In [5]:
df['Description'] = df['Description'].fillna('Unknown Product')


## Handling Missing Product Descriptions

The `Description` column contains a small number of missing values (~0.3% of total records).  
These records were **not removed** during data cleaning for the following business reasons:

- Missing product descriptions do **not impact revenue, sales trends, or customer behavior analysis**.
- Removing these rows would result in **loss of valid transaction and revenue data**, leading to under-reporting of key business KPIs.
- The dataset is transaction-level, and the financial integrity of each transaction is preserved even without a product description.

To maintain data completeness while supporting potential product-level analysis, missing values in the `Description` column were **replaced with `"Unknown Product"`**.

This approach ensures:
- No loss of transactional or revenue data
- Clear identification of records with incomplete product metadata
- Flexibility to include or exclude these records during product-specific analysis without affecting overall business insights


In [6]:
# Full dataset for sales & revenue analysis
df_sales = df.copy()

# Filtered dataset for customer behavior analysis
df_customers = df[df['CustomerID'].notna()]


## Handling Missing CustomerID Values

The `CustomerID` column contains a significant number of missing values, representing guest or unregistered customer transactions.

These records were **not removed from the dataset** because:
- They represent valid sales transactions and contribute to overall revenue.
- Removing them would lead to under-reporting of key sales and performance metrics.
- Customer identification is not required for sales, trend, or country-level analysis.

For analyses that require customer-level granularity (such as repeat purchases or customer behavior), a **filtered dataset excluding null CustomerID values** was created.

This approach ensures:
- Accurate and complete reporting of business performance
- Reliable customer analytics without distortion
- Clear separation between transactional and customer-focused analysis


In [7]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']


### Revenue Calculation
Revenue was derived as `Quantity × UnitPrice` to represent the financial value of each transaction, including the impact of returns.


In [9]:
df[['Quantity', 'UnitPrice', 'Revenue']].describe()


,Quantity,UnitPrice,Revenue
count,541909.000000,541909.000000,541909.000000
mean,9.552250,4.611114,17.987795
std,218.081158,96.759853,378.810824
min,-80995.000000,-11062.060000,-168469.600000
25%,1.000000,1.250000,3.400000
50%,3.000000,2.080000,9.750000
75%,10.000000,4.130000,17.400000
max,80995.000000,38970.000000,168469.600000


### Revenue Validation
A summary check was performed to ensure revenue values are reasonable and include both sales and returns.


In [10]:
df['IsReturn'] = df['Quantity'] < 0


### Returns Identification
An indicator was created to distinguish returned transactions from successful sales.


In [11]:
df_sales = df.copy()
df_customers = df[df['CustomerID'].notna()]


### Dataset Segmentation
The data was split to support both sales-level analysis and customer-level analysis without losing valid transactions.


In [12]:
df_sales.isnull().sum()


InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate    308950
UnitPrice           0
CustomerID     135080
Country             0
Revenue             0
IsReturn            0
dtype: int64

### Final Data Quality Check
- A final null-value review was conducted to confirm the dataset is ready for analysis.
- A large proportion of records contain invalid or missing InvoiceDate values, limiting their usability for time-based analysis.


In [14]:
df_sales_time = df_sales[df_sales['InvoiceDate'].notna()]


### InvoiceDate Handling
Rows with invalid InvoiceDate values were retained in the master dataset but excluded from time-based analysis to ensure accurate trend insights.


In [15]:
df_sales.to_csv("uk_ecommerce_bi_ready_all.csv", index=False)
df_sales_time.to_csv("uk_ecommerce_bi_ready_time.csv", index=False)


### Export BI-Ready Dataset
The cleaned dataset was exported for use in Power BI and Excel.


In [16]:
total_revenue = df_sales['Revenue'].sum()
total_transactions = df_sales.shape[0]
total_returns = df_sales[df_sales['IsReturn'] == True].shape[0]

total_revenue, total_transactions, total_returns


(np.float64(9747747.934), 541909, 10624)

## Overall Sales Performance
Basic KPIs were calculated to understand overall sales volume, revenue, and return impact.


In [17]:
revenue_by_country = (
    df_sales.groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
)

revenue_by_country.head(10)


Country
United Kingdom    8187806.364
Netherlands        284661.540
EIRE               263276.820
Germany            221698.210
France             197403.900
Australia          137077.270
Switzerland         56385.350
Spain               54774.580
Belgium             40910.960
Sweden              36595.910
Name: Revenue, dtype: float64

## Revenue by Country
Revenue contribution was analyzed at the country level to identify key markets.


In [19]:
# Create a clean dataset for time-based analysis
df_sales_time = df_sales[df_sales['InvoiceDate'].notna()].copy()

# Create Year-Month feature
df_sales_time['YearMonth'] = df_sales_time['InvoiceDate'].dt.to_period('M')

# Calculate monthly revenue
monthly_revenue = (
    df_sales_time
    .groupby('YearMonth')['Revenue']
    .sum()
    .sort_index()
)

monthly_revenue


YearMonth
2010-12    452134.28
2011-01    209688.48
2011-02    198039.06
2011-03    233725.22
2011-04    200102.66
2011-05    302095.85
2011-06    272881.43
2011-07    242129.70
2011-08    309443.04
2011-09    315612.94
2011-10    425926.66
2011-11    552831.24
2011-12    433668.01
Freq: M, Name: Revenue, dtype: float64

## Monthly Revenue Trend
Monthly revenue was aggregated using valid invoice dates to analyze business performance and seasonality over time.


In [20]:
return_revenue = df_sales[df_sales['IsReturn'] == True]['Revenue'].sum()
sales_revenue = df_sales[df_sales['IsReturn'] == False]['Revenue'].sum()

sales_revenue, return_revenue


(np.float64(10644560.424), np.float64(-896812.49))

## Return Impact Analysis
Returns were analyzed to measure their impact on total revenue.


In [21]:
top_products = (
    df_sales.groupby('Description')['Revenue']
    .sum()
    .sort_values(ascending=False)
)

top_products.head(10)


Description
DOTCOM POSTAGE                        206245.48
REGENCY CAKESTAND 3 TIER              164762.19
WHITE HANGING HEART T-LIGHT HOLDER     99668.47
PARTY BUNTING                          98302.98
JUMBO BAG RED RETROSPOT                92356.03
RABBIT NIGHT LIGHT                     66756.59
POSTAGE                                66230.64
PAPER CHAIN KIT 50'S CHRISTMAS         63791.94
ASSORTED COLOUR BIRD ORNAMENT          58959.73
CHILLI LIGHTS                          53768.06
Name: Revenue, dtype: float64

## Top Products by Revenue
Top-performing products were identified based on total revenue contribution.
